In [1]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

data = pd.read_csv("ab_test.csv")

display(data.head())
print(data.columns)
print(data.info())

,experiment_num;experiment_group;user_id;revenue
0,1;test;38456;520
1,1;control;13125924;806
2,1;control;9761984;0
3,1;test;11387012;208
4,1;test;18319648;104


Index(['experiment_num;experiment_group;user_id;revenue'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2836 entries, 0 to 2835
Data columns (total 1 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   experiment_num;experiment_group;user_id;revenue  2836 non-null   object
dtypes: object(1)
memory usage: 22.3+ KB
None


In [2]:
data = pd.read_csv("ab_test.csv", sep=";")

display(data.head())
print(data.columns)
data.info()

,experiment_num,experiment_group,user_id,revenue
0,1.0,test,38456.0,520.0
1,1.0,control,13125924.0,806.0
2,1.0,control,9761984.0,0.0
3,1.0,test,11387012.0,208.0
4,1.0,test,18319648.0,104.0


Index(['experiment_num', 'experiment_group', 'user_id', 'revenue'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2836 entries, 0 to 2835
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experiment_num    2835 non-null   float64
 1   experiment_group  2835 non-null   object 
 2   user_id           2835 non-null   float64
 3   revenue           2835 non-null   float64
dtypes: float64(3), object(1)
memory usage: 88.8+ KB


In [3]:
print(data["experiment_num"].unique())
print(data["experiment_group"].unique())

[ 1.  2.  3. nan]
['test' 'control' nan]


In [4]:
data = data.dropna()

print(data["experiment_num"].unique())
print(data["experiment_group"].unique())
data.info()

[1. 2. 3.]
['test' 'control']
<class 'pandas.core.frame.DataFrame'>
Index: 2835 entries, 0 to 2834
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experiment_num    2835 non-null   float64
 1   experiment_group  2835 non-null   object 
 2   user_id           2835 non-null   float64
 3   revenue           2835 non-null   float64
dtypes: float64(3), object(1)
memory usage: 110.7+ KB


In [5]:
# Приводим типы данных
data["experiment_num"] = data["experiment_num"].astype(int)
data["revenue"] = data["revenue"].astype(float)

print(data["experiment_num"].unique())
print(data["experiment_group"].unique())

display(data.head())
data.info()

#анализ ARPU:

summary = data.groupby(["experiment_num", "experiment_group"]).agg(
    users=("user_id", "nunique"),
    total_revenue=("revenue", "sum"),
    arpu=("revenue", "mean"),
    median_revenue=("revenue", "median")
).reset_index()

display(summary)

[1 2 3]
['test' 'control']


,experiment_num,experiment_group,user_id,revenue
0,1,test,38456.0,520.0
1,1,control,13125924.0,806.0
2,1,control,9761984.0,0.0
3,1,test,11387012.0,208.0
4,1,test,18319648.0,104.0


<class 'pandas.core.frame.DataFrame'>
Index: 2835 entries, 0 to 2834
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experiment_num    2835 non-null   int64  
 1   experiment_group  2835 non-null   object 
 2   user_id           2835 non-null   float64
 3   revenue           2835 non-null   float64
dtypes: float64(2), int64(1), object(1)
memory usage: 110.7+ KB


,experiment_num,experiment_group,users,total_revenue,arpu,median_revenue
0,1,control,465,335944.0,722.460215,104.0
1,1,test,480,319555.0,665.739583,104.0
2,2,control,465,327664.0,704.653763,74.0
3,2,test,480,159806.0,332.929167,52.0
4,3,control,465,308391.0,663.206452,4.0
5,3,test,480,479361.0,998.668750,156.0


In [6]:
from scipy.stats import ttest_ind

results = []

for exp in sorted(data["experiment_num"].unique()):
    exp_data = data[data["experiment_num"] == exp]
    
    control = exp_data[exp_data["experiment_group"] == "control"]["revenue"]
    test = exp_data[exp_data["experiment_group"] == "test"]["revenue"]
    
    control_arpu = control.mean()
    test_arpu = test.mean()
    
    difference = test_arpu - control_arpu
    percent_change = difference / control_arpu * 100
    
    t_stat, p_value = ttest_ind(test, control, equal_var=False)
    
    results.append({
        "experiment_num": exp,
        "control_users": len(control),
        "test_users": len(test),
        "control_arpu": control_arpu,
        "test_arpu": test_arpu,
        "difference": difference,
        "percent_change": percent_change,
        "p_value": p_value
    })

results_df = pd.DataFrame(results)

display(results_df)

# Добавить автоматический вывод
alpha = 0.05

for index, row in results_df.iterrows():
    print("Эксперимент", int(row["experiment_num"]))
    print("ARPU control:", round(row["control_arpu"], 2))
    print("ARPU test:", round(row["test_arpu"], 2))
    print("Разница:", round(row["difference"], 2))
    print("Изменение:", round(row["percent_change"], 2), "%")
    print("p-value:", round(row["p_value"], 4))
    
    if row["p_value"] < alpha:
        if row["difference"] > 0:
            print("Вывод: ARPU в test статистически значимо выше.")
            print("Рекомендация: изменение можно рассматривать к запуску.")
        else:
            print("Вывод: ARPU в test статистически значимо ниже.")
            print("Рекомендация: изменение запускать не стоит.")
    else:
        print("Вывод: статистически значимого различия ARPU нет.")
        print("Рекомендация: не делать вывод о победе test-группы.")
    
    print("-" * 50)

,experiment_num,control_users,test_users,control_arpu,test_arpu,difference,percent_change,p_value
0,1,465,480,722.460215,665.739583,-56.720632,-7.851039,0.688966
1,2,465,480,704.653763,332.929167,-371.724597,-52.752801,0.001128
2,3,465,480,663.206452,998.668750,335.462298,50.581881,0.060315


Эксперимент 1
ARPU control: 722.46
ARPU test: 665.74
Разница: -56.72
Изменение: -7.85 %
p-value: 0.689
Вывод: статистически значимого различия ARPU нет.
Рекомендация: не делать вывод о победе test-группы.
--------------------------------------------------
Эксперимент 2
ARPU control: 704.65
ARPU test: 332.93
Разница: -371.72
Изменение: -52.75 %
p-value: 0.0011
Вывод: ARPU в test статистически значимо ниже.
Рекомендация: изменение запускать не стоит.
--------------------------------------------------
Эксперимент 3
ARPU control: 663.21
ARPU test: 998.67
Разница: 335.46
Изменение: 50.58 %
p-value: 0.0603
Вывод: статистически значимого различия ARPU нет.
Рекомендация: не делать вывод о победе test-группы.
--------------------------------------------------
